# 第03課 - 代理設計模式

在本課中，我們將探討三個構建有效AI代理的基礎設計模式：

1. <strong>清晰的代理指令</strong> — 制定精確且定義角色的提示來引導代理行為
2. **使用Pydantic模型的結構化輸出** — 確保代理返回可預測且經過驗證的數據
3. <strong>單一職責代理</strong> — 設計專注且各司其職的代理

我們將把每個模式應用於一個<strong>旅遊目的地推薦</strong>場景，逐步建立一個可以建議目的地、檢查可用性並處理後勤的系統。


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 模式 1：清晰的代理指令

最具影響力的模式也是最簡單的：為你的代理撰寫清晰、詳細的指令。

良好的指令定義：
- 代理是<strong>誰</strong>（角色與語氣）
- 它應該做<strong>什麼</strong>（逐步職責）
- 它應該如何行事<strong>怎樣</strong>（限制與風格）

以下，我們創建了一個旅遊禮賓代理，具有明確的指令，塑造其所產生的每個回應。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 模式 2：使用 Pydantic 模型結構化輸出

自由格式文本對於對話很有用，但下游系統需要結構化資料。
通過將 **Pydantic 模型** 與 <strong>工具函數</strong> 配對，我們可以：

- 定義代理輸出的精確架構
- 自動驗證回應
- 可靠地將代理結果整合至應用邏輯

強制執行的關鍵是在運行代理時傳遞 `response_format`。這會強制
模型返回經驗證的 `TravelRecommendations` 物件（可於 `response.value` 獲取）
而非自由格式文本。`get_destination_details` 工具也返回類型化的
`DestinationRecommendation`，因此資料從頭到尾保持結構化。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## 模式 3：單一職責代理

複雜任務透過將工作分配給多個專注的代理，每個代理擁有單一職責，可獲得效益：

- 一位 <strong>目的地專家</strong>，了解地點與可用性
- 一位 <strong>物流規劃師</strong>，負責航班、酒店及行程安排

這呼應了軟體工程中的<em>關注點分離</em>原則 — 每個代理更容易被獨立測試、維護和改進。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 總結

本課程中，我們將三個代理設計模式應用於旅遊推薦情境：

| 模式 | 關鍵概念 | 好處 |
|---|---|---|
| <strong>清晰指示</strong> | 預先定義角色、責任及限制 | 穩定且符合品牌形象的代理行為 |
| <strong>結構化輸出</strong> | 使用 Pydantic 模型作為回應格式 | 經驗證、機器可讀的結果 |
| <strong>單一責任</strong> | 給予每個代理集中專注的任務 | 更易於測試、維護與組合 |

這些模式可以自然組合 —— 你可以在單一責任代理中結合清晰指示與結構化輸出，構建強健、適用於生產環境的系統。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
